# Cox Time-Varying Survival Model: Original BoL Features

This notebook is a more formal survival-model attempt using `lifelines`. It models time to **persistent delinquency/contact**, defined as the second observed delinquency/contact event year from 2000 to 2020. People without a second event are treated as censored at their last observed target year.

The model uses a time-varying Cox proportional hazards setup: each person contributes intervals over survey waves, and a feature is only available in an interval if it was measured before that interval's stop year.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from lifelines import CoxTimeVaryingFitter
from lifelines.utils import concordance_index

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

try:
    from IPython.display import display as safe_display
except Exception:
    def safe_display(x):
        print(x)


def find_project_dir(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "nlsy79_child_youngadult_selected_crime_features.csv").exists()
            and (candidate / "Tabular_based_models").exists()
            and (candidate / "BoL approach").exists()
        ):
            return candidate
    raise FileNotFoundError("Start Jupyter inside the Bol_Crime project folder or one of its subfolders.")

PROJECT_DIR = find_project_dir()
TABULAR_DIR = PROJECT_DIR / "Tabular_based_models"
MODEL_DIR = TABULAR_DIR / "survival_model"
OUT_DIR = MODEL_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_DIR / "nlsy79_child_youngadult_selected_crime_features.csv"
FEATURE_INDEX_PATH = PROJECT_DIR / "BoL approach" / "metadata_examples" / "child_crime_broad_persistent_feature_index.csv"
TARGETS_PATH = TABULAR_DIR / "data" / "targets" / "nlsy79_temporal_delinquency_targets_2000_2020.csv"
LR_IMPORTANCE_PATH = TABULAR_DIR / "logistic_regression" / "outputs" / "l2_logistic_regression_permutation_importance.csv"

TARGET = "later_persistent_delinquency_contact_2000_2020"
SECOND_EVENT_YEAR = "later_delinquency_contact_second_event_year_2000_2020"
LAST_OBSERVED_YEAR = "later_delinquency_contact_last_observed_year_2000_2020"
MISSING_CODES = {-1, -2, -3, -4, -5, -7}
WAVES = np.array([2000, 2002, 2004, 2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020])
RANDOM_SEED = 2026
TEST_SIZE = 0.30
FEATURE_LIMIT = 35
LOW_VARIANCE_THRESHOLD = 1e-8
COX_PENALIZER = 0.20

print("Project dir:", PROJECT_DIR)
print("Target file:", TARGETS_PATH)


## Load Target and Select Features

For this first Cox model, we use the original BoL feature index and select a compact subset of the strongest LR permutation-importance features. This keeps the Cox model stable and readable.


In [ ]:
data = pd.read_csv(DATA_PATH)
feature_index = pd.read_csv(FEATURE_INDEX_PATH)
targets = pd.read_csv(TARGETS_PATH)

available_features = [c for c in feature_index["csv_code"].tolist() if c in data.columns]
if LR_IMPORTANCE_PATH.exists():
    lr_importance = pd.read_csv(LR_IMPORTANCE_PATH)
    selected_features = [c for c in lr_importance["csv_code"].tolist() if c in available_features][:FEATURE_LIMIT]
else:
    selected_features = available_features[:FEATURE_LIMIT]

feature_year = feature_index.set_index("csv_code")["survey_year"].to_dict()
feature_meta = feature_index.set_index("csv_code")[["ref_id", "variable", "survey_year", "feature_group", "question"]]

person_df = data[["C0000100"] + selected_features].merge(
    targets[["C0000100", TARGET, SECOND_EVENT_YEAR, LAST_OBSERVED_YEAR]],
    on="C0000100",
    how="inner",
)
person_df = person_df[person_df[TARGET].notna() & person_df[LAST_OBSERVED_YEAR].notna()].copy()
person_df[TARGET] = person_df[TARGET].astype(int)

print("Eligible persons:", len(person_df))
print("Persistent event base rate:", round(person_df[TARGET].mean(), 3))
print("Selected features:", len(selected_features))
safe_display(feature_meta.loc[selected_features].reset_index().head(12))


## Build Time-Varying Cox Table

Each row is a risk interval. `start` and `stop` are measured as years since 1999, so the 2000 wave stops at 1, 2002 stops at 3, and so on. A feature is set missing if it was measured at or after the interval stop year. This avoids using future information inside the interval.


In [ ]:
def clean_person_feature_value(series):
    x = pd.to_numeric(series, errors="coerce")
    x = x.replace([np.inf, -np.inf], np.nan)
    x = x.mask(x.isin(MISSING_CODES), np.nan)
    return x

for col in selected_features:
    person_df[col] = clean_person_feature_value(person_df[col])

rows = []
for _, person in person_df.iterrows():
    child_id = int(person["C0000100"])
    event_year = person[SECOND_EVENT_YEAR]
    last_year = person[LAST_OBSERVED_YEAR]
    event_observed = int(person[TARGET])
    stop_year = int(event_year) if event_observed == 1 and pd.notna(event_year) else int(last_year)

    prev_stop = 0.0
    for wave in WAVES:
        if wave > stop_year:
            break
        row = {
            "C0000100": child_id,
            "start": prev_stop,
            "stop": float(wave - 1999),
            "wave": int(wave),
            "event": int(event_observed == 1 and wave == stop_year),
            TARGET: event_observed,
            SECOND_EVENT_YEAR: event_year,
            LAST_OBSERVED_YEAR: last_year,
        }
        for feature in selected_features:
            year = feature_year.get(feature)
            value = person[feature]
            if str(year) != "XRND":
                year_num = pd.to_numeric(pd.Series([year]), errors="coerce").iloc[0]
                if pd.notna(year_num) and wave <= year_num:
                    value = np.nan
            row[feature] = value
        rows.append(row)
        prev_stop = float(wave - 1999)

cox_long = pd.DataFrame(rows)
cox_long = cox_long[cox_long["stop"] > cox_long["start"]].copy()

print("Cox rows:", len(cox_long))
print("Events:", int(cox_long["event"].sum()))
print("Rows per wave:")
print(cox_long.groupby("wave")["event"].agg(rows="size", events="sum", event_rate="mean"))


## Train/Test Split and Preprocessing

The split is by person, not by row. Numeric features are median-imputed and standardized using training rows only. Low-variance columns are dropped before fitting.


In [ ]:
person_ids = person_df["C0000100"].astype(int).to_numpy()
person_labels = person_df[TARGET].astype(int).to_numpy()
train_ids, test_ids = train_test_split(
    person_ids,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=person_labels,
)
train_ids = set(train_ids.tolist())
test_ids = set(test_ids.tolist())

train_long = cox_long[cox_long["C0000100"].isin(train_ids)].copy()
test_long = cox_long[cox_long["C0000100"].isin(test_ids)].copy()

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

x_train = imputer.fit_transform(train_long[selected_features])
x_test = imputer.transform(test_long[selected_features])
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

variance = x_train.var(axis=0)
keep_mask = variance > LOW_VARIANCE_THRESHOLD
kept_features = [feature for feature, keep in zip(selected_features, keep_mask) if keep]
x_train = x_train[:, keep_mask]
x_test = x_test[:, keep_mask]

for j, feature in enumerate(kept_features):
    train_long[feature] = x_train[:, j]
    test_long[feature] = x_test[:, j]

cox_train = train_long[["C0000100", "start", "stop", "event"] + kept_features].copy()
cox_test = test_long[["C0000100", "start", "stop", "event", TARGET, SECOND_EVENT_YEAR, LAST_OBSERVED_YEAR] + kept_features].copy()

print("Train persons:", len(train_ids), "Test persons:", len(test_ids))
print("Train rows:", len(cox_train), "Test rows:", len(cox_test))
print("Kept features after variance filter:", len(kept_features))


## Fit Cox Time-Varying Model

The Cox model estimates relative hazard. Higher partial hazard means higher estimated risk of reaching the persistent-contact event sooner.


In [ ]:
ctv = CoxTimeVaryingFitter(penalizer=COX_PENALIZER)
ctv.fit(
    cox_train,
    id_col="C0000100",
    start_col="start",
    stop_col="stop",
    event_col="event",
    show_progress=False,
)

safe_display(ctv.summary.head(15))


## Evaluate Person-Level Risk Ranking

Cox models produce relative risk rather than direct class probabilities. For comparison, we aggregate each test person's interval hazards by taking the maximum partial hazard, then compute C-index and AUC against the persistent-event label.


In [ ]:
cox_test_eval = cox_test.copy()
cox_test_eval["partial_hazard"] = ctv.predict_partial_hazard(cox_test_eval[["C0000100", "start", "stop"] + kept_features]).to_numpy(dtype=float)

person_risk = (
    cox_test_eval
    .groupby("C0000100")
    .agg(
        y_true=(TARGET, "max"),
        duration=("stop", "max"),
        event=("event", "max"),
        max_partial_hazard=("partial_hazard", "max"),
        mean_partial_hazard=("partial_hazard", "mean"),
        n_intervals=("stop", "count"),
    )
    .reset_index()
)

# Concordance index expects larger scores to mean longer survival, so use the negative hazard.
c_index = concordance_index(
    person_risk["duration"].to_numpy(dtype=float),
    -person_risk["max_partial_hazard"].to_numpy(dtype=float),
    person_risk["event"].to_numpy(dtype=int),
)
auc = roc_auc_score(person_risk["y_true"].astype(int), person_risk["max_partial_hazard"])

risk_threshold = float(np.quantile(person_risk["max_partial_hazard"], 1 - person_risk["y_true"].mean()))
person_risk["prediction_matched_base_rate"] = (person_risk["max_partial_hazard"] >= risk_threshold).astype(int)
person_risk["correct_matched_base_rate"] = person_risk["prediction_matched_base_rate"] == person_risk["y_true"]

print("Person-level C-index:", round(c_index, 4))
print("Person-level AUC:", round(auc, 4))
print("Matched-base-rate threshold:", round(risk_threshold, 4))
print("Accuracy at matched-base-rate threshold:", round(person_risk["correct_matched_base_rate"].mean(), 4))
safe_display(person_risk.head(10))


## Save Outputs and Plot Coefficients

Positive coefficients increase the relative hazard of persistent delinquency/contact; negative coefficients decrease it. These are model-based associations, not causal effects.


In [ ]:
coef_df = ctv.summary.reset_index().rename(columns={"covariate": "csv_code"})
coef_df = coef_df.merge(
    feature_meta.reset_index(),
    on="csv_code",
    how="left",
)
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)

summary = {
    "model": "lifelines_cox_time_varying_persistent_second_event",
    "target": TARGET,
    "event_time": SECOND_EVENT_YEAR,
    "censor_time": LAST_OBSERVED_YEAR,
    "n_persons": int(len(person_df)),
    "n_train_persons": int(len(train_ids)),
    "n_test_persons": int(len(test_ids)),
    "n_rows_train": int(len(cox_train)),
    "n_rows_test": int(len(cox_test)),
    "feature_limit": FEATURE_LIMIT,
    "n_kept_features": int(len(kept_features)),
    "cox_penalizer": COX_PENALIZER,
    "test_size": TEST_SIZE,
    "random_seed": RANDOM_SEED,
    "person_level_c_index": float(c_index),
    "person_level_auc": float(auc),
    "matched_base_rate_threshold": risk_threshold,
    "matched_base_rate_accuracy": float(person_risk["correct_matched_base_rate"].mean()),
}

coef_df.to_csv(OUT_DIR / "cox_time_varying_coefficients.csv", index=False)
person_risk.to_csv(OUT_DIR / "cox_time_varying_person_risk.csv", index=False)
(OUT_DIR / "cox_time_varying_summary.json").write_text(json.dumps(summary, indent=2))

safe_display(coef_df.head(15))

plot_df = coef_df.head(12).copy()
labels = plot_df["variable"].fillna(plot_df["csv_code"]).astype(str) + " (" + plot_df["survey_year"].fillna("time").astype(str) + ")"
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(len(plot_df)), plot_df["abs_coef"])
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("absolute Cox coefficient")
ax.set_title("Cox Time-Varying Survival: Top Coefficients")
fig.tight_layout()
fig.savefig(OUT_DIR / "cox_time_varying_top_coefficients.png", dpi=200, bbox_inches="tight")
plt.show()
